In [1]:
import pandas as pd
import numpy as np
import torch
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error




In [2]:
train_df = pd.read_parquet("../data/model_ready/train_final.parquet")
train_df.head()

,LocationNormalized,Company,Category,SalaryNormalized,SourceName,ContractType_full_time,ContractType_part_time,ContractType_nan,ContractTime_contract,ContractTime_permanent,...,bert_758,bert_759,bert_760,bert_761,bert_762,bert_763,bert_764,bert_765,bert_766,bert_767
0,25825.692857,33829.545455,35838.268332,25000,30552.161848,False,False,True,False,True,...,-0.263804,-0.600368,0.162118,-0.765770,-0.889339,0.575798,-0.865606,-0.637418,0.212452,1.037239
1,33313.998318,33829.545455,35838.268332,30000,30552.161848,False,False,True,False,True,...,-0.145878,-0.560483,0.216945,-0.769155,-0.861016,0.580105,-0.806191,-0.617189,0.151145,0.937013
2,32016.803198,33829.545455,35838.268332,30000,30552.161848,False,False,True,False,True,...,-0.170805,-0.498284,0.088202,-0.759696,-0.875579,0.539534,-0.776423,-0.678497,0.097144,0.990441
3,33159.031250,33829.545455,35838.268332,27500,30552.161848,False,False,True,False,True,...,-0.258938,-0.590406,0.133464,-0.781387,-0.892273,0.525957,-0.801068,-0.639413,0.246292,1.027552
4,33159.031250,33829.545455,35838.268332,25000,30552.161848,False,False,True,False,True,...,-0.254813,-0.692882,0.136335,-0.654949,-0.846868,0.531075,-0.827661,-0.609582,0.220572,0.988587


In [3]:
print("Loading the Parquet masterpieces...")
train_df = pd.read_parquet("../data/model_ready/train_final.parquet")
valid_df = pd.read_parquet("../data/model_ready/valid_final.parquet")
test_df = pd.read_parquet("../data/model_ready/test_final.parquet")

# 1. Split into Features (X) and Target (y)
print("Splitting X and y...")
target_col = 'SalaryNormalized'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)
y_test_log = np.log1p(y_test)

Loading the Parquet masterpieces...
Splitting X and y...


In [4]:
print("Force-feeding 32-bit floats to PyTorch...")

# 1. Force the features into pure float32 arrays
X_train_matrix = np.array(X_train.values, dtype=np.float32)
X_valid_matrix = np.array(X_valid.values, dtype=np.float32)
X_test_matrix  = np.array(X_test.values, dtype=np.float32)

# 2. Force the targets into 2D float32 arrays
y_train_matrix = np.array(y_train_log.values, dtype=np.float32).reshape(-1, 1)
y_valid_matrix = np.array(y_valid_log.values, dtype=np.float32).reshape(-1, 1)

print(f"Train Matrix Type: {X_train_matrix.dtype} | Target Type: {y_train_matrix.dtype}")
print("If this doesn't say float32, I will literally eat my own code.")

Force-feeding 32-bit floats to PyTorch...
Train Matrix Type: float32 | Target Type: float32
If this doesn't say float32, I will literally eat my own code.


In [5]:
print("Cranking the graphics to Ultra... RIP GPU.")

tabnet_model = TabNetRegressor(
    n_d=128,                 # Width of decision layer (Up from 32)
    n_a=128,                 # Width of attention layer (Up from 32)
    n_steps=8,               # How deep the network goes (Up from 4)
    gamma=1.5,               # More aggressive feature selection
    n_independent=4,         # More independent layers
    n_shared=4,              # More shared layers
    seed=42,
    optimizer_fn=torch.optim.AdamW, # Upgraded to AdamW for better weight decay
    optimizer_params=dict(lr=1e-2, weight_decay=1e-4),
    scheduler_params={"step_size": 10, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax',   # Stricter attention mechanism
    device_name='cuda'
)

# And change your batch size in the .fit() method to this:
tabnet_model.fit(
    X_train=X_train_matrix, y_train=y_train_matrix,
    eval_set=[(X_valid_matrix, y_valid_matrix)],
    eval_name=['valid'],
    eval_metric=['mae'],
    max_epochs=20,          # Let it cook longer
    patience=10,
    batch_size=4096,        # MASSIVE chunks of data at once (Eats VRAM)
    virtual_batch_size=1024, # Huge ghost batches
    num_workers=0,
    drop_last=False
)

Cranking the graphics to Ultra... RIP GPU.


c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 11.03594| valid_mae: 0.49602 |  0:00:31s
epoch 1  | loss: 3.59832 | valid_mae: 0.29728 |  0:01:02s
epoch 2  | loss: 2.80215 | valid_mae: 0.40395 |  0:01:34s
epoch 3  | loss: 0.94538 | valid_mae: 0.2928  |  0:02:04s
epoch 4  | loss: 0.53002 | valid_mae: 0.31798 |  0:02:35s
epoch 5  | loss: 0.185   | valid_mae: 0.19013 |  0:03:05s
epoch 6  | loss: 0.13441 | valid_mae: 0.19536 |  0:03:34s
epoch 7  | loss: 0.22441 | valid_mae: 0.3439  |  0:04:03s
epoch 8  | loss: 0.16076 | valid_mae: 0.40134 |  0:04:35s
epoch 9  | loss: 0.13976 | valid_mae: 0.21645 |  0:05:08s
epoch 10 | loss: 0.0976  | valid_mae: 0.20306 |  0:05:39s
epoch 11 | loss: 0.10964 | valid_mae: 0.23467 |  0:06:19s
epoch 12 | loss: 0.10547 | valid_mae: 0.21386 |  0:06:47s
epoch 13 | loss: 0.12615 | valid_mae: 0.28487 |  0:07:16s
epoch 14 | loss: 0.09945 | valid_mae: 0.20977 |  0:07:53s
epoch 15 | loss: 0.0981  | valid_mae: 0.25874 |  0:08:25s

Early stopping occurred at epoch 15 with best_epoch = 5 and best_valid_

c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [ ]:

print("Predicting...")
# TabNet spits out a 2D array, we must flatten it back to 1D
y_pred_log = tabnet_model.predict(X_test_matrix).flatten()

# Reverse the log cheat code
y_pred_real = np.expm1(y_pred_log)

# Flex the Metrics
mae = mean_absolute_error(y_test, y_pred_real)
r2 = r2_score(y_test, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_real))
print("\n" + "="*30)
print("🧠 TABNET (GOOGLE BRAIN) RESULTS 🧠")
print("="*30)
print(f"R2 Score: {r2:.4f}")
print(f"MAE:      £{mae:.2f}")
print(f"RMSE:     £{rmse:.2f}")
print("="*30 + "\n")

Predicting...

🧠 TABNET (GOOGLE BRAIN) RESULTS 🧠
R2 Score: 0.5426
MAE:      £6260.45
RMSE:           £8162.80

